# Análise de terremotos (USGS, M ≥ 2.5)

Dados limpos por `src/clean.py`. Perguntas:
1. Quantos terremotos ocorrem por dia e há tendência?
2. Qual a distribuição de magnitudes (Gutenberg-Richter)?
3. Quais regiões concentram mais eventos?
4. Onde ocorrem os maiores terremotos (M ≥ 5)?
5. Como se distribui a profundidade e ela se relaciona com a magnitude?
6. Existe padrão por hora do dia (UTC)?

In [1]:
import pandas as pd
import plotly.express as px
df = pd.read_csv('../data/processed/earthquakes.csv')
df['time'] = pd.to_datetime(df['time'], utc=True, format='ISO8601')
print(df.shape, df['time'].min(), '->', df['time'].max())
df.head()

(4236, 15) 2026-09-07 00:06:24.092000+00:00 -> 2026-11-05 01:07:21.383000+00:00


,id,time,date,hour,latitude,longitude,depth,mag,magType,mag_class,depth_class,place,region,status,updated
0,us7000tesb,2026-09-07 00:06:24.092000+00:00,2026-09-07,0,36.1143,70.8202,121.473,3.6,mb,3-4,intermediaria (70-300 km),"77 km N of Pārūn, Afghanistan",Afghanistan,reviewed,2026-10-06 22:34:42.040000+00:00
1,aka2026rrycnr,2026-09-07 00:19:38.107000+00:00,2026-09-07,0,61.5470,-146.4870,20.100,2.8,ml,<3,rasa (<70 km),"46 km N of Valdez, Alaska",Alaska,reviewed,2026-09-12 02:41:39.040000+00:00
2,us7000thec,2026-09-07 00:56:51.088000+00:00,2026-09-07,0,-8.3078,120.6867,10.000,4.3,mb,4-5,rasa (<70 km),"41 km NE of Ruteng, Indonesia",Indonesia,reviewed,2026-10-06 22:30:37.040000+00:00
3,us7000tet2,2026-09-07 01:02:45.050000+00:00,2026-09-07,1,51.6698,176.5445,35.000,3.7,mb,3-4,rasa (<70 km),"264 km ESE of Attu Station, Alaska",Alaska,reviewed,2026-09-26 23:08:21.040000+00:00
4,us7000tesk,2026-09-07 01:30:00.764000+00:00,2026-09-07,1,43.7329,-128.3141,10.000,3.9,mb,3-4,rasa (<70 km),off the coast of Oregon,off the coast of Oregon,reviewed,2026-09-27 00:36:38.040000+00:00


## 1. Eventos por dia

In [2]:
daily = df.groupby('date').size().reset_index(name='eventos').iloc[:-1]  # descarta o último dia (UTC), ainda incompleto
print('média/dia:', round(daily['eventos'].mean(), 1))
px.line(daily, x='date', y='eventos', markers=True, title='Terremotos por dia')

média/dia: 71.7


## 2. Distribuição de magnitudes
Esperado: contagem cai ~10x a cada +1 de magnitude (escala log).

In [3]:
fig = px.histogram(df, x='mag', nbins=40, log_y=True, title='Histograma de magnitudes (eixo Y log)')
fig

## 3. Regiões com mais eventos

In [4]:
top = df['region'].value_counts().head(15).reset_index()
px.bar(top, x='count', y='region', orientation='h', title='Top 15 regiões').update_yaxes(autorange='reversed')

## 4. Maiores terremotos (M ≥ 5)

In [5]:
big = df[df['mag'] >= 5]
print(len(big), 'eventos M>=5')
display(big.sort_values('mag', ascending=False)[['time','mag','place','depth']].head(10))
px.scatter_geo(big, lat='latitude', lon='longitude', size='mag', color='mag', hover_name='place', projection='natural earth', title='Eventos M ≥ 5')

347 eventos M>=5


,time,mag,place,depth
2490,2026-10-09 17:56:05.537000+00:00,7.7,"10 km WSW of Pitaloza Arriba, Panama",10.0
3503,2026-10-22 14:33:40.289000+00:00,7.0,"189 km ENE of Nagqu, China",18.0
3055,2026-10-16 03:35:32.092000+00:00,6.9,"72 km NE of Ozernovskiy, Russia",135.0
3605,2026-10-24 02:31:28.641000+00:00,6.9,"197 km WNW of Port-Olry, Vanuatu",10.0
2890,2026-10-13 12:23:07.248000+00:00,6.8,central Mid-Atlantic Ridge,10.0
2862,2026-10-13 05:44:41.603000+00:00,6.7,central Mid-Atlantic Ridge,10.0
1397,2026-09-25 21:23:03.906000+00:00,6.7,"74 km ENE of Tadine, New Caledonia",10.0
2549,2026-10-09 20:25:38.424000+00:00,6.6,"13 km WSW of Río Grande, Panama",10.0
774,2026-09-17 14:19:52.234000+00:00,6.5,"170 km W of Nikolski, Alaska",98.0
363,2026-09-11 21:23:55.815000+00:00,6.5,"121 km NNE of Teluknaga, Indonesia",372.0


## 5. Profundidade × magnitude

In [6]:
print(df['depth_class'].value_counts(normalize=True).round(3))
print('correlação depth~mag:', round(df['depth'].corr(df['mag']), 3))
px.box(df, x='depth_class', y='mag', title='Magnitude por classe de profundidade')

depth_class
rasa (<70 km)                0.780
intermediaria (70-300 km)    0.182
profunda (>300 km)           0.038
Name: proportion, dtype: float64
correlação depth~mag: 0.14


## 6. Padrão por hora do dia (UTC)
Terremotos são naturais, então esperamos distribuição ~uniforme; variações podem refletir detecção.

In [7]:
hourly = df.groupby('hour').size().reset_index(name='eventos')
px.bar(hourly, x='hour', y='eventos', title='Eventos por hora UTC')

## Conclusões
Respostas calculadas a partir dos dados (atualizam a cada execução).

In [8]:
from IPython.display import Markdown, display
import numpy as np

n_dias = df['date'].nunique()
d = daily['eventos']
slope = np.polyfit(range(len(d)), d, 1)[0]
top3 = df['region'].value_counts().head(3)
top5 = df[df['mag'] >= 5]['region'].value_counts().head(3)
top_pct = top3.iloc[0] / len(df)
maior = df.loc[df['mag'].idxmax()]
dc = df['depth_class'].value_counts(normalize=True)
med = df.groupby('depth_class')['mag'].median()
corr = df['depth'].corr(df['mag'])
h = hourly['eventos']
cv_obs = h.std() / h.mean()
cv_poisson = 1 / np.sqrt(h.mean())
frac_4 = (df['mag'] >= 4).mean()
r1 = top3.index[0]
med_r1 = df[df['region'] == r1]['mag'].median()
med_resto = df[df['region'] != r1]['mag'].median()
dia_pico = daily.loc[d.idxmax(), 'date']

display(Markdown(f"""
**Período analisado:** {df['date'].min()} a {df['date'].max()} ({n_dias} dias, {len(df):,} eventos M ≥ 2,5).

1. **Eventos por dia:** média de {d.mean():.1f}/dia (mín. {d.min()}, máx. {d.max()} em {dia_pico}). \
Tendência linear de {slope:+.2f} eventos/dia a cada dia. O dia de pico {'coincide' if dia_pico == maior['date'] else 'não coincide'} \
com o dia do maior terremoto do período ({maior['date']}). O último dia (UTC), ainda incompleto, foi excluído.
2. **Magnitudes:** mediana {df['mag'].median():.2f}; {frac_4:.0%} dos eventos têm M ≥ 4. \
O decaimento esperado de ~10× por unidade de magnitude (Gutenberg-Richter) não aparece abaixo de M≈4: o catálogo é **incompleto** \
fora de regiões bem instrumentadas, o que cria uma distribuição com dois picos. Evidência: a mediana em {r1} é \
{med_r1:.1f}, contra {med_resto:.1f} no resto do mundo.
3. **Regiões:** {r1} lidera com {top3.iloc[0]:,} eventos ({top_pct:.0%} do total), seguida por \
{top3.index[1]} ({top3.iloc[1]:,}) e {top3.index[2]} ({top3.iloc[2]:,}). Esse domínio reflete em parte a maior \
sensibilidade da rede sísmica local, não só maior atividade.
4. **Maiores eventos:** {(df['mag'] >= 5).sum()} eventos M ≥ 5; o maior foi M {maior['mag']:.1f} \
({maior['place']}, {maior['date']}). Regiões com mais eventos M ≥ 5: {', '.join(f'{r} ({v})' for r, v in top5.items())}.
5. **Profundidade:** {dc.get('rasa (<70 km)', 0):.0%} dos eventos são rasos (<70 km), {dc.get('intermediaria (70-300 km)', 0):.0%} intermediários e \
{dc.get('profunda (>300 km)', 0):.0%} profundos. Correlação profundidade × magnitude = {corr:.2f} (fraca); \
mediana da magnitude: rasos {med.get('rasa (<70 km)', float('nan')):.2f} vs. intermediários {med.get('intermediaria (70-300 km)', float('nan')):.2f}.
6. **Hora do dia (UTC):** variação entre horas (CV = {cv_obs:.1%}) é compatível com o ruído aleatório esperado \
para contagens (≈{cv_poisson:.1%}); não há padrão horário relevante, como esperado para um fenômeno natural.

**Limitações:** completude do catálogo desigual entre regiões; `region` é extraída do texto de `place`; janela de poucos meses.
"""))



**Período analisado:** 2026-09-07 a 2026-11-05 (60 dias, 4,236 eventos M ≥ 2,5).

1. **Eventos por dia:** média de 71.7/dia (mín. 42, máx. 137 em 2026-10-09). Tendência linear de -0.43 eventos/dia a cada dia. O dia de pico coincide com o dia do maior terremoto do período (2026-10-09). O último dia (UTC), ainda incompleto, foi excluído.
2. **Magnitudes:** mediana 4.10; 55% dos eventos têm M ≥ 4. O decaimento esperado de ~10× por unidade de magnitude (Gutenberg-Richter) não aparece abaixo de M≈4: o catálogo é **incompleto** fora de regiões bem instrumentadas, o que cria uma distribuição com dois picos. Evidência: a mediana em Alaska é 2.8, contra 4.4 no resto do mundo.
3. **Regiões:** Alaska lidera com 1,291 eventos (30% do total), seguida por Indonesia (223) e Vanuatu (166). Esse domínio reflete em parte a maior sensibilidade da rede sísmica local, não só maior atividade.
4. **Maiores eventos:** 347 eventos M ≥ 5; o maior foi M 7.7 (10 km WSW of Pitaloza Arriba, Panama, 2026-10-09). Regiões com mais eventos M ≥ 5: Vanuatu (63), Indonesia (30), Panama (27).
5. **Profundidade:** 78% dos eventos são rasos (<70 km), 18% intermediários e 4% profundos. Correlação profundidade × magnitude = 0.14 (fraca); mediana da magnitude: rasos 4.07 vs. intermediários 4.20.
6. **Hora do dia (UTC):** variação entre horas (CV = 7.0%) é compatível com o ruído aleatório esperado para contagens (≈7.5%); não há padrão horário relevante, como esperado para um fenômeno natural.

**Limitações:** completude do catálogo desigual entre regiões; `region` é extraída do texto de `place`; janela de poucos meses.
